## 📓 Notebook 04: Classifiers — Naive Bayes, Logistic Regression, LinearSVC

**Skills practiced:** Training three model types on identical TF-IDF features, timing `.fit()`, comparing accuracy and macro-F1.

**Goal:** Train all three classifiers on the same features and build one comparison table covering accuracy, macro-F1, and training time.

In [101]:
import pandas as pd
import time
import joblib

from scipy.sparse import load_npz
from scipy.special import softmax

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC



In [102]:
x_train=load_npz('../data/processed/x_train.npz')
x_test=load_npz('../data/processed/x_test.npz')
y_train=pd.read_csv('../data/processed/y_train.csv').squeeze('columns')
y_test=pd.read_csv('../data/processed/y_test.csv').squeeze('columns')

In [103]:
s_time=time.time()
m=MultinomialNB().fit(x_train,y_train)
e_time=time.time()

proba = m.predict_proba(x_test)

m1_scores={
        "model name": 'MultinomialNB',
        "accuracy": accuracy_score(y_test, m.predict(x_test)),
        "macro-f1": f1_score(y_test, m.predict(x_test), average='macro'), 
        "roc_auc": roc_auc_score(y_test, proba, multi_class='ovr', average='macro'),
        "trained-time": (e_time-s_time)%60}

In [104]:
s_time=time.time()
m2=LogisticRegression(max_iter=1000).fit(x_train,y_train)
e_time=time.time()

proba = m2.predict_proba(x_test)

m2_scores={
        "model name": 'LogisticRegression',
        "accuracy": accuracy_score(y_test, m2.predict(x_test)),
        "macro-f1": f1_score(y_test, m2.predict(x_test), average='macro'), 
        "roc_auc": roc_auc_score(y_test, proba, multi_class='ovr', average='macro'),
        "trained-time": (e_time-s_time)%60}

In [105]:
s_time=time.time()
m3=LinearSVC().fit(x_train,y_train)
e_time=time.time()

scores = m3.decision_function(x_test)
proba = softmax(scores, axis=1) #rescales each row so the values are positive and sum to 1.

m3_scores={
        "model name": 'LinearSVC',
        "accuracy": accuracy_score(y_test, m3.predict(x_test)),
        "macro-f1": f1_score(y_test, m3.predict(x_test), average='macro'),
        "roc_auc": roc_auc_score(y_test, proba, multi_class='ovr', average='macro'),
        "trained-time": (e_time-s_time)%60}

In [106]:
table = pd.DataFrame([m1_scores, m2_scores, m3_scores])
print(table)

           model name  accuracy  macro-f1   roc_auc  trained-time
0       MultinomialNB    0.9060  0.902799  0.996704      0.131813
1  LogisticRegression    0.9792  0.979018  0.998877      0.688735
2           LinearSVC    0.9796  0.979420  0.998784      8.318705


In [107]:
joblib.dump(m3, '../models/LinearSVC_model.joblib')

['../models/LinearSVC_model.joblib']